# CodeBlue | Regional ocean-data visualisation

**Portable analysis notebook** for the files produced by `download_cmems_bgc.py`, `download_woa23.py` and `download_oras5.py`.

**Scientific interpretation:** WOA23 monthly climatologies describe the mean seasonal cycle (not individual years), and the WOA23 annual climatology is used for deeper layers. No synthetic pre-1993 time series are generated. ORAS5 native T/U/V grids remain independent. Spatial means here are **unweighted arithmetic means of valid horizontal grid cells**, not cell-area-weighted averages; layer averages are weighted by depth-cell thickness. Standard deviations describe **spatial dispersion**, not temporal uncertainty.

Run the cells in order. **Only configuration cell 1 needs editing.** Large historical archives can be slow; begin with a short period.

## 1. Configure input directories and analysis subregion

The directories below are relative to the notebook's parent repository directory. Open the notebook from its `visualization/` folder with Jupyter's working directory set there; `REPO_ROOT` resolves the repo from either the root or this folder. To analyse data stored elsewhere, set the three directory paths explicitly. **Bounds below specify the analysis subregion, with no additional 1° padding** (the padding applies only during downloading).

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

HERE = Path.cwd().resolve()
REPO_ROOT = HERE if (HERE / "download").exists() else HERE.parent
DATA_ROOT = REPO_ROOT / "data"
CMEMS_DIR = DATA_ROOT / "CMEMS"
WOA_DIR = DATA_ROOT / "WOA"
ORAS5_DIR = DATA_ROOT / "ORAS5"
OUTPUT_DIR = REPO_ROOT / "figures"

# USER-SELECTED ANALYSIS REGION (inside the downloaded geographic coverage):
WEST, EAST = -10.0, -8.0
SOUTH, NORTH = 45.0, 47.0

# Optional period restrictions for plots (YYYY-MM). None = use all available months.
START_MONTH = "2020-01"
END_MONTH = "2020-12"

# WOA23 -> Copernicus comparison. WOA23 typically uses micromol/kg.
REFERENCE_DENSITY = 1025.0  # kg m-3; approximate conversion (not in-situ density)
SAVE_FIGURES = False
DPI = 160

# BGC depth intervals: nutrients 0–800, 800–3000, >3000; oxygen 0–1500, >1500.
BGC_LAYERS = {
    "no3": [(0, 800), (800, 3000), (3000, np.inf)],
    "po4": [(0, 800), (800, 3000), (3000, np.inf)],
    "si":  [(0, 800), (800, 3000), (3000, np.inf)],
    "o2":  [(0, 1500), (1500, np.inf)],
}
PHY_LAYERS = [(0, 100), (100, 1000), (1000, np.inf)]
BGC_NAMES = {"no3":"Nitrate", "po4":"Phosphate", "si":"Silicate", "o2":"Dissolved oxygen"}
PHY_NAMES = {"votemper":"Temperature", "vosaline":"Salinity", "vozocrtx":"Zonal velocity",
             "vomecrty":"Meridional velocity", "sossheig":"Sea surface height"}

assert -180 <= WEST < EAST <= 180 and -90 <= SOUTH < NORTH <= 90, "Invalid bounds (antimeridian unsupported)"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
print("Repository:", REPO_ROOT)
print("Analysis region (no extra padding):", (WEST,EAST,SOUTH,NORTH))
print("Inputs:", CMEMS_DIR, WOA_DIR, ORAS5_DIR, sep="\n  ")

## 2. Locate datasets and inspect coverage

Products can be absent: visualisations for missing products are skipped. Missing months are never silently converted to zero.

In [ ]:
CMEMS_FILES = sorted(CMEMS_DIR.glob("*/BGC_??????.nc"))
ORAS5_FILES = sorted(ORAS5_DIR.glob("*/ORAS5_??????.nc"))
WOA_MONTHLY = {m: WOA_DIR / f"WOA_{m:02d}.nc" for m in range(1,13)}
WOA_ANNUAL = WOA_DIR / "WOA_ANNUAL.nc"
print("Copernicus monthly files:", len(CMEMS_FILES))
print("ORAS5 monthly files:", len(ORAS5_FILES))
print("WOA23 monthly climatologies:", sum(p.exists() for p in WOA_MONTHLY.values()), "/12")
print("WOA23 annual climatology:", WOA_ANNUAL.exists())

def file_month(path):
    import re
    match = re.search(r"_(\d{6})\.nc$", path.name)
    return pd.Timestamp(f"{match.group(1)[:4]}-{match.group(1)[4:]}-01") if match else None

def period_files(files):
    lo = pd.Timestamp(START_MONTH + "-01") if START_MONTH else pd.Timestamp.min
    hi = pd.Timestamp(END_MONTH + "-01") if END_MONTH else pd.Timestamp.max
    return [p for p in files if file_month(p) is not None and lo <= file_month(p) <= hi]

CMEMS_FILES = period_files(CMEMS_FILES)
ORAS5_FILES = period_files(ORAS5_FILES)
print("Period-filtered months: BGC", len(CMEMS_FILES), "/ PHY", len(ORAS5_FILES))

## 3. Shared regional selection and averaging

For 1D lat/lon (Copernicus, WOA23), select by coordinate. For 2D native ORAS5 geographic coordinates, form a mask independently for each field. Ensure the requested region intersects the data; only existing wet cells contribute to statistics. On the native ORAS5 grid, we use the variable-specific `nav_lat_...` / `nav_lon_...` coordinates written by the downloader.

In [ ]:
def regional_field(ds, var):
    """Return (dataarray, horizontal_dims) cropped/masked to user region."""
    if var not in ds:
        raise KeyError(f"{var} not in {list(ds.data_vars)}")
    a = ds[var]
    if "latitude" in a.dims and "longitude" in a.dims:
        sub = a.where((ds.latitude >= SOUTH)&(ds.latitude <= NORTH), drop=True)
        sub = sub.where((sub.longitude >= WEST)&(sub.longitude <= EAST), drop=True)
        horizontal = ("latitude","longitude")
    elif "lat" in a.dims and "lon" in a.dims:
        sub = a.where((ds.lat >= SOUTH)&(ds.lat <= NORTH), drop=True)
        sub = sub.where((sub.lon >= WEST)&(sub.lon <= EAST), drop=True)
        horizontal = ("lat","lon")
    else:
        latkey = next((k for k in ["nav_lat_"+var, "latitude_"+var, "lat_"+var, "nav_lat"] if k in ds), None)
        lonkey = next((k for k in ["nav_lon_"+var, "longitude_"+var, "lon_"+var, "nav_lon"] if k in ds), None)
        if latkey is None or lonkey is None:
            raise ValueError(f"No usable coordinates for {var}; dataset variables: {list(ds.variables)}")
        lat, lon = ds[latkey], ((ds[lonkey]+180)%360)-180
        if lat.ndim != 2 or lon.ndim != 2 or lat.dims != lon.dims:
            raise ValueError(f"Expected 2D ORAS5 coordinates for {var}")
        horizontal = tuple(lat.dims)
        mask = (lat>=SOUTH)&(lat<=NORTH)&(lon>=WEST)&(lon<=EAST)
        sub = a.where(mask, drop=True)
    if any(sub.sizes.get(d,0)==0 for d in horizontal):
        raise ValueError(f"Requested region does not overlap {var} data")
    return sub, horizontal

def time_dim(a):
    return next((d for d in a.dims if "time" in d.lower()), None)

def depth_dim(a):
    return next((d for d in a.dims if "depth" in d.lower() or d.lower() in ("lev","level")), None)

def unit_to_mmol_m3(a):
    """Convert known nutrient/oxygen units; fail loudly on unexpected units."""
    s = str(a.attrs.get("units","")).lower().replace(" ","").replace("μ","µ")
    if any(k in s for k in ("mmolm-3","mmol/m3","mmolm^-3")): fac=1.
    elif any(k in s for k in ("µmolkg-1","umolkg-1","µmol/kg","umol/kg","micromol/kg","micromoles_per_kilogram","micromolesperkilogram","micromolkg-1")): fac=REFERENCE_DENSITY/1000
    elif any(k in s for k in ("µmoll-1","umol/l","µmol/l")): fac=1.
    elif any(k in s for k in ("molm-3","mol/m3")): fac=1000.
    else: raise ValueError(f"Unknown BGC units: {a.attrs.get('units')!r}")
    return a * fac

def depth_weights(depth, minimum, maximum):
    z = np.asarray(depth, dtype=float)
    if len(z)<2 or np.any(np.diff(z)<=0):
        raise ValueError("Depth must be strictly increasing with >=2 levels")
    edges = np.r_[0,(z[:-1]+z[1:])/2,z[-1]+(z[-1]-z[-2])/2]
    return np.maximum(0, np.minimum(edges[1:],maximum)-np.maximum(edges[:-1],minimum))

def layer_statistics(a, horizontal, minimum, maximum):
    """Depth-weight per cell, then spatial mean and population SD. Retains time axis."""
    zdim = depth_dim(a)
    if zdim is None:
        column = a
    else:
        weights = depth_weights(a[zdim].values,minimum,maximum)
        w = xr.DataArray(weights,dims=(zdim,),coords={zdim:a[zdim]})
        valid = w.where(a.notnull(),0)
        denom = valid.sum(zdim)
        column = (a.fillna(0)*valid).sum(zdim)/denom.where(denom>0)
    return column.mean(horizontal,skipna=True), column.std(horizontal,skipna=True,ddof=0)

def profile(a,horizontal):
    td=time_dim(a)
    if td: a=a.mean(td,skipna=True)
    return a.mean(horizontal,skipna=True)

def maybe_save(name):
    if SAVE_FIGURES:
        plt.savefig(OUTPUT_DIR/name,dpi=DPI,bbox_inches="tight")
    plt.show()

def nice_layer(z):
    a,b=z
    return f"{a:g}–{b:g} m" if np.isfinite(b) else f">{a:g} m"

print("Functions ready")

## 4. Geographic coverage maps

Shows original downloaded grid points and the selected analysis subregion. Without Cartopy, this is a longitude–latitude diagnostic rather than a coastline map.

In [ ]:
from matplotlib.patches import Rectangle

def coverage_points(ds,var):
    if "latitude" in ds[var].dims:
        lo,la=np.meshgrid(ds.longitude.values,ds.latitude.values)
    elif "lat" in ds[var].dims:
        lo,la=np.meshgrid(ds.lon.values,ds.lat.values)
    else:
        lo=ds["nav_lon_"+var].values
        la=ds["nav_lat_"+var].values
    lo=((lo+180)%360)-180
    return lo,la

sources=[]
for title,paths,var in [("Copernicus BGC",CMEMS_FILES,"no3"),("ORAS5",ORAS5_FILES,"votemper"),
                        ("WOA23",[WOA_MONTHLY[1]] if WOA_MONTHLY[1].exists() else [],"no3")]:
    if paths:
        with xr.open_dataset(paths[0]) as ds:
            try: sources.append((title,*coverage_points(ds,var)))
            except Exception as exc: warnings.warn(f"Coverage {title}: {exc}")
if sources:
    fig,axes=plt.subplots(1,len(sources),figsize=(5.5*len(sources),4.5),squeeze=False)
    for ax,(name,lon,lat) in zip(axes.ravel(),sources):
        ax.scatter(lon.ravel(),lat.ravel(),s=2,alpha=.35,rasterized=True)
        ax.add_patch(Rectangle((WEST,SOUTH),EAST-WEST,NORTH-SOUTH,fill=False,edgecolor="red",lw=2))
        ax.set(xlabel="Longitude (°E)",ylabel="Latitude (°N)",title=name)
        ax.grid(alpha=.2)
    fig.tight_layout(); maybe_save("coverage.png")
else: print("No files available for coverage figure")

## 5. Vertical profiles and WOA23–Copernicus comparison

BGC: compare January WOA23 monthly climatology (upper layers) and WOA23 annual values (deep layers) against **the first available Copernicus January**, if present. This is a *climatology vs a specific-year* comparison, not an estimate of trends. Plot an annual WOA23 profile independently if monthly January is absent. ORAS5: mean profile for the earliest available month.

In [ ]:
def woa_composite_profile(var,month=1):
    """Monthly shallow profile + annual deep profile, without vertical interpolation."""
    monthly=WOA_MONTHLY[month]
    if not monthly.exists() and not WOA_ANNUAL.exists(): return None
    pieces=[]
    shallow_limit = 1500 if var=="o2" else 800
    if monthly.exists():
        with xr.open_dataset(monthly) as ds:
            if var in ds:
                a,hor=regional_field(ds,var); p=profile(unit_to_mmol_m3(a),hor).load()
                pieces.append(p.where(p.depth<=shallow_limit,drop=True))
    if WOA_ANNUAL.exists():
        with xr.open_dataset(WOA_ANNUAL) as ds:
            if var in ds:
                a,hor=regional_field(ds,var); p=profile(unit_to_mmol_m3(a),hor).load()
                if pieces: p=p.where(p.depth>shallow_limit,drop=True)
                pieces.append(p)
    if not pieces: return None
    return xr.concat(pieces,dim="depth").sortby("depth")

january=[p for p in CMEMS_FILES if file_month(p).month==1]
for v,label in BGC_NAMES.items():
    fig,ax=plt.subplots(figsize=(5.5,7))
    w=woa_composite_profile(v)
    if w is not None: ax.plot(w.values,w.depth.values,label="WOA23 (monthly shallow + annual deep)",lw=2)
    if january:
        with xr.open_dataset(january[0]) as ds:
            if v in ds:
                a,hor=regional_field(ds,v); p=profile(unit_to_mmol_m3(a),hor)
                ax.plot(p.values,p[depth_dim(p)].values,label=f"Copernicus {file_month(january[0]):%Y-%m}")
    ax.invert_yaxis(); ax.set(xlabel=f"{label} (mmol m⁻³)",ylabel="Depth (m)",title=label)
    ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); maybe_save(f"BGC_profile_{v}.png")

if ORAS5_FILES:
    with xr.open_dataset(ORAS5_FILES[0]) as ds:
        for v,label in PHY_NAMES.items():
            if v not in ds or v=="sossheig":continue
            a,hor=regional_field(ds,v); p=profile(a,hor); zd=depth_dim(p)
            if zd is None:continue
            fig,ax=plt.subplots(figsize=(5.5,7));ax.plot(p.values,p[zd].values)
            ax.invert_yaxis();ax.set(xlabel=f"{label} ({a.attrs.get('units','')})",ylabel="Depth (m)",title=f"ORAS5 {file_month(ORAS5_FILES[0]):%Y-%m}")
            ax.grid(alpha=.25);fig.tight_layout();maybe_save(f"ORAS5_profile_{v}.png")

## 6. Process downloaded months into regional statistics

For each Copernicus day: mean profile over the subregion (Hovmöller input); depth-weighted mean of every horizontal column followed by the spatial mean and ±1 spatial SD for each layer. For ORAS5, equivalent statistics are calculated monthly on each field's native grid. Calculations are **lazy until extraction of compact arrays**, and one input file is processed at a time. For large historical datasets, narrow the analysis period in cell 1.

In [ ]:
def extract_time_index(ds,a,path,expected):
    td=time_dim(a)
    if td is None: return pd.DatetimeIndex([file_month(path)])
    try: index=pd.DatetimeIndex(pd.to_datetime(ds[td].values))
    except Exception:
        # Some ORAS5 NetCDF files use undecodable calendar times: filename defines monthly index.
        if expected=="monthly" and a.sizes[td]==1:return pd.DatetimeIndex([file_month(path)])
        raise
    if expected=="monthly" and len(index)!=1:
        raise ValueError(f"Expected one ORAS5 time step: {path}")
    return index

def process_product(files, variables, layer_map, bgc=False, expected="monthly"):
    output={}
    for var in variables:
        dates=[]; means=[]; stds=[]; hov=[]; reference_depth=None; units=None
        layers=layer_map[var]
        for idx,path in enumerate(files):
            try:
                with xr.open_dataset(path) as ds:
                    if var not in ds: continue
                    a,hor=regional_field(ds,var)
                    units=a.attrs.get("units","")
                    if bgc: a=unit_to_mmol_m3(a);units="mmol m-3"
                    td=time_dim(a)
                    t=extract_time_index(ds,a,path,expected)
                    if td is None: a=a.expand_dims(_sample_time=t)
                    # The native time dimension is retained; each file is read independently.
                    ntime=len(t)
                    zdim=depth_dim(a)
                    if zdim is not None:
                        z=np.asarray(a[zdim].values,dtype=float)
                        q=profile_by_time=a.mean(hor,skipna=True)
                        q=np.asarray(q.transpose(time_dim(q),zdim).values,dtype=float)
                        if reference_depth is None: reference_depth=z
                        elif not np.array_equal(reference_depth,z):
                            # Explicit remap of the *mean profile only* onto reference depths.
                            q=np.vstack([np.interp(reference_depth,z,row,left=np.nan,right=np.nan) for row in q])
                    else:
                        q=np.full((ntime,0),np.nan)
                        if reference_depth is None:reference_depth=np.array([])
                    mm=np.full((ntime,len(layers)),np.nan)
                    ss=np.full_like(mm,np.nan)
                    for k,(z0,z1) in enumerate(layers):
                        m,s=layer_statistics(a,hor,z0,z1)
                        mm[:,k]=np.asarray(m.values).reshape(-1)
                        ss[:,k]=np.asarray(s.values).reshape(-1)
                    dates.extend(t);means.extend(mm);stds.extend(ss);hov.extend(q)
            except Exception as exc:
                warnings.warn(f"Skipped {path.name}/{var}: {exc}")
        if dates:
            dates=pd.DatetimeIndex(dates)
            order=np.argsort(dates.values)
            output[var]={"time":dates[order],"mean":np.asarray(means)[order],
                         "std":np.asarray(stds)[order],"hov":np.asarray(hov)[order],
                         "depth":reference_depth,"layers":layers,"units":units}
            print(f"{var}: {len(dates)} time steps, {len(reference_depth)} depth levels")
        else: print(f"{var}: no data in chosen region/period")
    return output

BGC_RESULTS=process_product(CMEMS_FILES, list(BGC_NAMES), BGC_LAYERS,bgc=True,expected="daily") if CMEMS_FILES else {}
PHY_LAYER_MAP={v: ([(0,np.inf)] if v=="sossheig" else PHY_LAYERS) for v in PHY_NAMES}
PHY_RESULTS=process_product(ORAS5_FILES,list(PHY_NAMES),PHY_LAYER_MAP,expected="monthly") if ORAS5_FILES else {}

## 7. Depth-layer time series with spatial ±1 SD

Copernicus BGC is **daily**; ORAS5 is **monthly**. Shaded bands are the across-cell spatial standard deviation after vertical averaging, *not* temporal variability. WOA23 is displayed separately in the next section, as climatological observations are not independent annual measurements.

In [ ]:
def plot_series(results, labels, prefix):
    for var,obj in results.items():
        fig,ax=plt.subplots(figsize=(13,4.5))
        for i,depth_range in enumerate(obj["layers"]):
            color=f"C{i}"
            mean=obj["mean"][:,i];sd=obj["std"][:,i]
            ax.plot(obj["time"],mean,lw=1.5,color=color,label=nice_layer(depth_range))
            ax.fill_between(obj["time"],mean-sd,mean+sd,alpha=.18,color=color)
        ax.set(title=f"{prefix} — {labels[var]} | selected region",ylabel=obj["units"] or var,xlabel="Time")
        ax.grid(alpha=.2);ax.legend();fig.tight_layout();maybe_save(f"{prefix}_layers_{var}.png")
plot_series(BGC_RESULTS,BGC_NAMES,"CMEMS")
plot_series(PHY_RESULTS,PHY_NAMES,"ORAS5")

## 8. WOA23 climatological seasonal cycles — monthly shallow, annual deep

For nutrients use monthly climatologies at **0–800 m** and the annual climatology at deeper layers. For oxygen use monthly climatologies at **0–1500 m** and annual climatology below. Deep annual climatology is repeated *across the 12 calendar months only to compare layer statistics across the mean seasonal cycle*; this does not represent observed monthly variations. Units are converted to mmol m⁻³ using the reference-density approximation.

In [ ]:
def woa_layer_cycle(var):
    layers=BGC_LAYERS[var]
    means=np.full((12,len(layers)),np.nan)
    stds=np.full_like(means,np.nan)
    cutoff=1500 if var=="o2" else 800
    for i,(lo,hi) in enumerate(layers):
        annual=lo>=cutoff
        paths=[WOA_ANNUAL]*12 if annual else [WOA_MONTHLY[m] for m in range(1,13)]
        for mi,p in enumerate(paths):
            if not p.exists():continue
            try:
                with xr.open_dataset(p) as ds:
                    a,hor=regional_field(ds,var)
                    a=unit_to_mmol_m3(a)
                    m,s=layer_statistics(a,hor,lo,hi)
                    means[mi,i]=float(m.values)
                    stds[mi,i]=float(s.values)
            except Exception as exc:warnings.warn(f"WOA {p.name}/{var}: {exc}")
    return means,stds

if any(p.exists() for p in WOA_MONTHLY.values()) or WOA_ANNUAL.exists():
    month_index=np.arange(1,13)
    for v,label in BGC_NAMES.items():
        mm,ss=woa_layer_cycle(v)
        fig,ax=plt.subplots(figsize=(10,4))
        for i,layer in enumerate(BGC_LAYERS[v]):
            ax.plot(month_index,mm[:,i],marker="o",color=f"C{i}",label=nice_layer(layer))
            ax.fill_between(month_index,mm[:,i]-ss[:,i],mm[:,i]+ss[:,i],color=f"C{i}",alpha=.18)
        ax.set(xticks=month_index,xlabel="Climatological calendar month",ylabel="mmol m⁻³",title=f"WOA23 — {label} climatological seasonal cycle")
        ax.grid(alpha=.2);ax.legend();fig.tight_layout();maybe_save(f"WOA_cycle_{v}.png")

## 9. Hovmöller diagrams: time × depth (0 m at the top)

Color scales use robust 2–98th percentiles **within each figure**, avoiding a misleading global scale across different variables. The WOA23 seasonal cycle is plotted separately from the time-resolved products. ORAS5 SSH is excluded because it has no vertical coordinate.

In [ ]:
def plot_hov(times,z,values,title,name,units="", max_depth=None):
    if len(z)==0 or np.asarray(values).size==0:return
    z=np.asarray(z,dtype=float);values=np.asarray(values,dtype=float)
    if max_depth is not None:
        sel=z<=max_depth;z=z[sel];values=values[:,sel]
    if not len(z) or not np.isfinite(values).any():return
    good=values[np.isfinite(values)]
    vmin,vmax=np.nanpercentile(good,[2,98])
    if np.isclose(vmin,vmax):vmin,vmax=float(good.min())-0.5,float(good.max())+0.5
    fig,ax=plt.subplots(figsize=(13,6))
    x=mdates.date2num(pd.to_datetime(times).to_pydatetime())
    mesh=ax.pcolormesh(x,z,np.ma.masked_invalid(values.T),shading="auto",cmap="viridis",vmin=vmin,vmax=vmax)
    ax.invert_yaxis();ax.xaxis_date();ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.set(title=title,xlabel="Time",ylabel="Depth (m)")
    fig.colorbar(mesh,ax=ax,label=units);fig.autofmt_xdate();fig.tight_layout();maybe_save(name)

for v,item in BGC_RESULTS.items():
    plot_hov(item["time"],item["depth"],item["hov"],f"Copernicus BGC — {BGC_NAMES[v]}",f"CMEMS_Hov_{v}.png",item["units"])
for v,item in PHY_RESULTS.items():
    if v!="sossheig":
        plot_hov(item["time"],item["depth"],item["hov"],f"ORAS5 — {PHY_NAMES[v]}",f"ORAS5_Hov_{v}.png",item["units"])

if all(p.exists() for p in WOA_MONTHLY.values()):
    for v,label in BGC_NAMES.items():
        monthly=[];zs=None
        for m in range(1,13):
            with xr.open_dataset(WOA_MONTHLY[m]) as ds:
                a,hor=regional_field(ds,v);q=profile(unit_to_mmol_m3(a),hor)
                cutoff=1500 if v=="o2" else 800
                q=q.where(q.depth<=cutoff,drop=True)
                if zs is None:zs=q.depth.values
                if not np.array_equal(zs,q.depth.values):raise ValueError("WOA23 monthly depth grids differ")
                monthly.append(q.values)
        plot_hov(pd.date_range("2000-01-01",periods=12,freq="MS"),zs,np.array(monthly),
                 f"WOA23 — {label} monthly climatology (not a year 2000 observation)",f"WOA_Hov_{v}.png","mmol m⁻³")

## 10. Optional quality-control summary

Reports finite values in the depth–time matrices. Large missing areas may reflect bathymetry, land masks, missing downloaded months, or unavailable depth levels. Check their cause before interpreting a figure.

In [ ]:
for name,items in [("CMEMS",BGC_RESULTS),("ORAS5",PHY_RESULTS)]:
    for var,item in items.items():
        hov=item["hov"]
        finite=int(np.isfinite(hov).sum())
        print(f"{name:6} {var:10} {len(item['time']):5} time steps | {len(item['depth']):3} levels | {finite:7} finite time-depth means")
print("Done. Figures directory:",OUTPUT_DIR, "| Saving enabled:", SAVE_FIGURES)

## Reproducibility and limitations

- **Sources and paths:** `data/CMEMS/YYYY/BGC_YYYYMM.nc`, `data/WOA/WOA_MM.nc`, `data/WOA/WOA_ANNUAL.nc`, and `data/ORAS5/YYYY/ORAS5_YYYYMM.nc` are the outputs of the repository's downloader scripts.
- **No geographic padding** is added during analysis: the user sets a smaller region than the region already downloaded.
- **WOA23 annual vs monthly:** monthly nutrients extend to 800 m, monthly O₂ to 1500 m; the original annual climatology supplies deep layers. Annual values plotted across calendar months do not add seasonal information.
- **WOA23 units:** when input is µmol kg⁻¹, conversion to mmol m⁻³ uses reference density 1025 kg m⁻³; this is an approximation, not a measured in-situ density conversion.
- **Spatial averages:** unweighted arithmetic mean over valid cells. Vertical layer means use midpoint-derived depth-thickness weights. Spatial standard deviation is computed across vertically averaged columns, with population normalization (`ddof=0`).
- **Horizontal geometry:** selected rectangular subregion, not a specific CROCO open-boundary segment or coast-following line. To reproduce the four original MANGAE2500 sectors, run separate regions or implement geometry masks.
- **ORAS5:** each variable is selected using its own stored 2-D geographic coordinates. Any changes in ORAS5 grid naming should be checked against the NetCDF. Native fields are not horizontally interpolated.
- **Hovmöller:** cell means are plotted on native depth levels. Only if source monthly depth coordinates change within a product, the *regional mean vertical profile* is interpolated to the first file's depths.
- **Performance:** Processing long archives can take time. Set a short `START_MONTH`/`END_MONTH` for initial QC; output files are not modified.
- **Scientific scope:** descriptive diagnostics only; figures do not perform independent product bias correction or model boundary-condition interpolation.
